# Performance matrix and the 10 km buffer
-------

Configurations whose main nested-CV run has not finished are shown as `pending`; re-running
`scripts/run_performance_matrix.py` and `scripts/run_gap_analysis.py` fills them in.
Source: `results/performance_matrix/` and `results/spatial_buffer/`.

## Figure output

This notebook exports three publication figures as PDFs. Flip a flag below to `False` to skip a figure (it is then neither written nor drawn inline); set `SAVE_FIGURES = False` to skip all figure output.

In [ ]:
SAVE_FIGURES = True  # master switch for all figure output
FIGURE_TOGGLES = {
    "fig_6_feature_importance_by_predictor_stack": True,  # XGBoost gain importance per predictor stack
    # per-stack PR-AUC, EO gain over baseline, EO representation contrasts
    "eo_signal_and_representation_contrasts": True,
    "eo_signal_and_representation_contrasts_0km": True,  # same figure, 0 km arm only
    # full PR-AUC matrix: feature set x architecture x buffer
    "fig_4_prauc_by_feature_set_and_architecture": True,
    # spatial-buffer justification: single-panel performance change + three-panel diagnostic
    "fig_5_performance_change_vs_buffer": True,
    "fig_s7_prauc_vs_buffer": True,
    "fig_s6_spatial_buffer_justification": True,
}

In [ ]:
NOTEBOOK = "009_performance_matrix_and_buffers"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FormatStrFormatter, MultipleLocator

from utils.paths import get_project_paths
from utils.style import get_figure_size, save_figure, use_publication_style
from utils.terminology import ARCHITECTURES, FEATURE_SETS, PALETTE_CATEGORICAL, SEMANTIC_COLOURS

# Publication style: Charis SIL (bundled in assets/fonts), applied via the shared
# utils.style helper. Set once here; every figure below inherits it.
use_publication_style()
plt.rcParams.update(
    {
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.titlepad": 6,
    }
)

# Point figures (1-3): grey = within AOI (0 km), with the buffered arm below it. Each point
# figure is produced at both the 10 km headline buffer (green) and a 20 km robustness buffer
# (magenta); the two are separate figures sharing the same paired-arm layout.
HEADLINE_COLOUR = "0.45"
BUFFER_COLOUR = SEMANTIC_COLOURS["ogf"]
BUFFER_COLOUR_20 = PALETTE_CATEGORICAL["magenta"]
ARM_STYLE = {
    0: (HEADLINE_COLOUR, "No spatial buffer"),
    10: (BUFFER_COLOUR, "10 km spatial buffer"),
    20: (BUFFER_COLOUR_20, "20 km spatial buffer"),
}
# Marker shape per arm, so the arms stay distinguishable without colour: a diamond within
# the AOI, a square for the spatially buffered arms.
ARM_MARKER = {0: "D", 10: "s", 20: "s"}
# Gap-sweep figures (4-5): the spatial buffer arm vs a matched random control.
GAP_BUFFERED = PALETTE_CATEGORICAL["blue"]
GAP_CONTROL = PALETTE_CATEGORICAL["orange"]
# Buffered-arm colour per feature set for the multi-stack performance-change figure. TESSERA is
# the headline (blue); the extra stacks reuse the feature-importance figure's hues.
GAP_FS_COLOUR = {
    "baseline_tessera": PALETTE_CATEGORICAL["blue"],
    "baseline_alphaearth": PALETTE_CATEGORICAL["magenta"],
    "baseline_conventional_eo": PALETTE_CATEGORICAL["teal"],
    "baseline": "0.3",
    "xy_coords": PALETTE_CATEGORICAL["orange"],
}
# Marker shape per feature set for the buffer figures, so the lines can be told apart
# without colour.
GAP_FS_MARKER = {
    "baseline_tessera": "o",
    "baseline_alphaearth": "s",
    "baseline_conventional_eo": "^",
    "baseline": "D",
    "xy_coords": "x",
}
# Full feature-set names for the buffer-figure legends (Baseline-first ordering).
GAP_FS_LABEL = {
    "baseline": "Baseline",
    "baseline_conventional_eo": "Baseline + conventional EO",
    "baseline_alphaearth": "Baseline + AlphaEarth",
    "baseline_tessera": "Baseline + TESSERA",
    "xy_coords": "Coordinate-only control (x, y)",
}
STARVE_THRESHOLD = 0.20  # a fold "collapses" when under 20% of its gap-0 training survives

# Short, presentation-ready names for the two-line axis labels.
SHORT_FS = {
    "baseline": "Baseline",
    "baseline_conventional_eo": "conventional EO",
    "baseline_alphaearth": "AlphaEarth",
    "baseline_tessera": "TESSERA",
    "xy_coords": "Coordinate-only (x, y)",
}
SHORT_ARCH = {
    "xgboost": "XGBoost",
    "cnn_3x3": "CNN 3x3",
    "cnn_5x5": "CNN 5x5",
    "cnn_7x7": "CNN 7x7",
}
# Panel titles have the room to spell the CNN kernel with a multiplication sign; the
# compact SHORT_ARCH form stays in the axis and contrast labels, where space is tighter.
TITLE_ARCH = {
    "xgboost": "XGBoost",
    "cnn_3x3": "CNN 3 \u00d7 3",
    "cnn_5x5": "CNN 5 \u00d7 5",
    "cnn_7x7": "CNN 7 \u00d7 7",
}


def _short(name):
    """Short label for a config name (architecture for arch__fs keys, else feature set)."""
    return SHORT_ARCH.get(name.split("__")[0], name) if "__" in name else SHORT_FS.get(name, name)


def per_config_label(feature_set):
    """Two-line feature-set label: 'Baseline' or 'Baseline\n+ X'."""
    return "Baseline" if feature_set == "baseline" else f"Baseline\n+ {SHORT_FS[feature_set]}"


def contrast_label(key):
    """Two-line contrast label 'A\n\u2212 B' from a '<a> - <b>' contrast key."""
    set_a, set_b = key.split(" - ")
    return f"{_short(set_a)}\n\u2212 {_short(set_b)}"


def contrast_label_full(key, *, wrap_at=None):
    """Two-line contrast label with full, square-bracketed names: '[A]\n\u2212 [B]'.

    Handles feature-set keys ('baseline_tessera - baseline') and architecture keys
    ('cnn_3x3__baseline_tessera - xgboost__baseline_tessera'), where the architecture is the part
    that varies and the shared feature set is named in the panel subtitle instead.
    ``wrap_at`` splits a bracketed name over two lines at its ' + ' when the inner name is
    longer than that many characters, for panels with a narrow left margin.
    """
    set_a, set_b = key.split(" - ")

    def _full(name):
        if "__" in name:
            return SHORT_ARCH.get(name.split("__")[0], name)
        return FEATURE_SETS.get(name, SHORT_FS.get(name, name))

    def _bracket(text):
        # Wrap at the ' + ' so a long name occupies two narrow lines instead of one wide one;
        # used for the panel whose left margin is the figure's narrowest.
        if wrap_at is not None and len(text) > wrap_at and " + " in text:
            head, tail = text.split(" + ", 1)
            return f"[{head} +\n{tail}]"
        return f"[{text}]"

    return f"{_bracket(_full(set_a))}\n\u2212 {_bracket(_full(set_b))}"


def save_if_enabled(fig, name, **kwargs):
    """Write ``fig`` as a PDF only when ``SAVE_FIGURES`` and the figure's toggle are
    on (see the "Figure output" cell at the top); otherwise close it unsaved."""
    stem = str(name).rsplit("/", 1)[-1]
    if SAVE_FIGURES and FIGURE_TOGGLES.get(stem, False):
        return save_figure(fig, name, **kwargs)
    plt.close(fig)
    return None


paths = get_project_paths()
# The newest run that has written its matrix: a run still computing its buffered refits has
# a folder but no CSV yet.
pm_run = sorted(
    p
    for p in (paths.results / "performance_matrix").iterdir()
    if p.is_dir() and (p / "performance_matrix.csv").is_file()
)[-1]
matrix = pd.read_csv(pm_run / "performance_matrix.csv")
fs_contrasts = pd.read_csv(pm_run / "feature_set_contrasts.csv")
emb_contrasts = pd.read_csv(pm_run / "embedding_contrasts.csv")
arch_contrasts = pd.read_csv(pm_run / "architecture_contrasts.csv")

# gap_analysis.csv is written only when every fold finishes, so it marks a complete run;
# until one exists (a crashed or in-progress sweep has none) the Why-10 km figures are pending.
buffer_root = paths.results / "spatial_buffer"
gap_runs = sorted(
    p
    for p in (buffer_root.iterdir() if buffer_root.is_dir() else [])
    if p.is_dir() and (p / "gap_analysis.csv").is_file()
)
# Runs grouped by feature set (dir name <ts>__xgboost__<feature_set>). A feature set may have
# more than one run - e.g. a buffered-only sweep and a later control-only sweep - so keep the
# whole list and merge their arms below rather than keeping only the newest.
gap_runs_by_fs = {}
for p in gap_runs:
    if p.name.count("__") >= 2:
        gap_runs_by_fs.setdefault(p.name.split("__", 2)[2], []).append(p)
# TESSERA is the headline arm: it drives Table 4, the retention chart and the starve logic.
tessera_runs = gap_runs_by_fs.get("baseline_tessera")
gap_run = tessera_runs[-1] if tessera_runs else (gap_runs[-1] if gap_runs else None)
if gap_run is None:
    print("[load] no completed spatial_buffer run; the Why-10 km figures will show pending.")
    gap_detail = gap_counts = gap_summary = None
    gap_detail_by_fs = {}
else:
    gap_detail_path = gap_run / "gap_detail.csv"
    gap_detail = pd.read_csv(gap_detail_path) if gap_detail_path.is_file() else None
    gap_counts = pd.read_csv(gap_run / "gap_parcel_counts.csv")
    gap_summary = pd.read_csv(gap_run / "gap_analysis.csv")
    # Per-feature-set gap detail for the performance-change figure. A feature set's buffered and
    # control arms may live in separate runs (buffered-only + control-only), so combine them,
    # newest run winning per arm, and measure each feature set against its OWN control.
    gap_detail_by_fs = {}
    for fs, runs_fs in gap_runs_by_fs.items():
        arms_latest = {}
        for p in runs_fs:  # gap_runs sorted ascending -> a newer run overwrites per arm
            detail_path = p / "gap_detail.csv"
            if not detail_path.is_file():
                continue
            for arm, sub in pd.read_csv(detail_path).groupby("arm"):
                arms_latest[arm] = sub
        if arms_latest:
            gap_detail_by_fs[fs] = pd.concat(arms_latest.values(), ignore_index=True)

METRICS = [
    "pr_auc",
    "roc_auc",
    "f1_outer",
    "precision_outer",
    "recall_outer",
    "f1_inner",
    "precision_inner",
    "recall_inner",
]
# Display name: the conventional-EO feature set reads "conventional EO" everywhere
# (never plain "EO" or capitalised "Conventional"); patch the shared label locally.
FEATURE_SETS = {**FEATURE_SETS, "baseline_conventional_eo": "Baseline + conventional EO"}
FS_ORDER = list(FEATURE_SETS)
ARCH_ORDER = list(ARCHITECTURES)


def fmt(value, lo=None, hi=None, places=3):
    """Format a value with an optional interval; 'pending' for NaN."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "pending"
    if lo is None or np.isnan(lo):
        return f"{value:.{places}f}"
    return f"{value:+.{places}f} [{lo:+.{places}f}, {hi:+.{places}f}]"


def arm_legend(fig, buffer_km=10, ncol=2, arms=None, **legend_kw):
    """One shared within-AOI / spatially-gapped legend (0 km vs ``buffer_km``) below the figure.

    ``arms`` restricts the entries to the buffers a figure actually shows."""
    handles = [
        Line2D(
            [0], [0], marker=ARM_MARKER[km], color=ARM_STYLE[km][0], lw=0, label=ARM_STYLE[km][1]
        )
        for km in ((0, buffer_km) if arms is None else arms)
    ]
    fig.legend(handles=handles, loc="outside lower center", ncol=ncol, frameon=False, **legend_kw)


print(f"[load] matrix {pm_run.name}; gap {gap_run.name if gap_run is not None else 'pending'}")
print(f"[load] trained matrix cells: {(matrix['folds_used'] > 0).sum()}/{len(matrix)}")
matrix[["feature_set", "architecture", "buffer_km", "folds_used"]].head(8)

In [ ]:
import geopandas as gpd

from utils.bootstrap import percentile_interval
from utils.reporting import metric_matrix_row

# Extend the performance matrix with accuracy, TSS and MCC (at the nested-CV inner threshold) as
# three extra metric families, and append the coordinate-only control as a feature set. The extras
# are functions of the four confusion counts, bootstrapped over the same 20 spatial blocks and seed
# as the matrix's own intervals, so one draw-multiplicity matrix yields every replicate. The control
# is scored exactly like every other config (full CIs); fmt falls back to point-only wherever a CI
# is NaN, so the tables update dynamically. Reloaded from the CSV so re-running stays idempotent.
matrix = pd.read_csv(pm_run / "performance_matrix.csv")
_src = paths.results / "main_nested_cv"
_lab = gpd.read_file(paths.labels / "ogf_reference_labels_partitioned.gpkg")
_lab = _lab[_lab["ogf"].notna()]
_ba = pd.DataFrame(
    {
        "parcel_id": _lab["parcel_id"].astype(np.int64),
        "bootstrap_id": _lab["bootstrap_id"].astype(np.int64),
    }
)
_cache = paths.cache / "performance_matrix_buffered"
_reps, _seed, _nblocks = 5000, 42, int(_ba["bootstrap_id"].nunique())
_draws = np.stack(
    [
        np.bincount(rng.integers(0, _nblocks, size=_nblocks), minlength=_nblocks)
        for rng in map(np.random.default_rng, np.random.SeedSequence(_seed).spawn(_reps))
    ]
).astype(np.float64)
EXTRA_METRICS = ["accuracy", "TSS", "MCC"]
MATRIX_METRICS = METRICS + EXTRA_METRICS


def _skill_counts(tp, tn, fp, fn):
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid="ignore", divide="ignore"):
        return {
            "accuracy": (tp + tn) / (tp + tn + fp + fn),
            "TSS": tp / (tp + fn) + tn / (tn + fp) - 1.0,
            "MCC": np.where(den > 0, (tp * tn - fp * fn) / den, 0.0),
        }


def _headline_run(arch, fs):
    for run in sorted(_src.glob(f"*__{arch}__{fs}"), reverse=True):
        folds = [
            f
            for f in range(1, 7)
            if (run / f"parcel_predictions_fold{f}.parquet").is_file()
            and (run / f"hp_trials_fold{f}.csv").is_file()
        ]
        if len(folds) == 6:
            return run, folds
    return None, []


def _inner_from_trials(run, folds):
    thr = {}
    for f in folds:
        trials = pd.read_csv(run / f"hp_trials_fold{f}.csv")
        complete = trials[trials["state"] == "COMPLETE"]
        thr[f] = float(complete.loc[complete["pr_auc"].idxmax(), "inner_parcel_threshold"])
    return thr


def _fold_frames(fs, arch, km, run, folds):
    if km == 0:
        return {f: pd.read_parquet(run / f"parcel_predictions_fold{f}.parquet") for f in folds}
    cdir = _cache / f"{arch}__{fs}__{run.name}__{km}km"
    return {f: pd.read_parquet(cdir / f"buffered_fold{f}.parquet") for f in folds}


def _inner_skill(frames, inner):
    """{'accuracy'|'TSS'|'MCC': (point, ci_lo, ci_hi)} at the inner threshold from pooled OOF."""
    pooled = pd.concat(
        [fr.assign(outer_fold=int(f)) for f, fr in sorted(frames.items())], ignore_index=True
    )
    y = pooled["y_true"].to_numpy().astype(np.int64)
    p = pooled["p_mean"].to_numpy().astype(np.float64)
    per = np.array([inner[int(f)] for f in pooled["outer_fold"].to_numpy()], dtype=np.float64)
    s = (p >= per).astype(np.float64)
    merged = pooled.merge(_ba, on="parcel_id", how="left")
    has = merged["bootstrap_id"].notna().to_numpy()
    bid = merged["bootstrap_id"].to_numpy()[has].astype(np.int64)
    yb, sb = y > 0.5, s > 0.5
    point = _skill_counts(
        *[np.float64((a & b).sum()) for a, b in ((sb, yb), (~sb, ~yb), (sb, ~yb), (~sb, yb))]
    )
    _, idx = np.unique(bid, return_inverse=True)
    yh, sh = yb[has], sb[has]
    tp, tn, fp, fn = (
        _draws @ np.bincount(idx, weights=w.astype(float), minlength=_nblocks)
        for w in (sh & yh, ~sh & ~yh, sh & ~yh, ~sh & yh)
    )
    valid = ((tp + fn) > 0) & ((tn + fp) > 0)
    boot = _skill_counts(tp, tn, fp, fn)
    out = {}
    for name in EXTRA_METRICS:
        arr = np.where(valid, boot[name], np.nan)
        lo, hi = percentile_interval(arr, level=0.95)
        out[name] = (float(point[name]), lo, hi)
    return out


for _name in EXTRA_METRICS:
    for _suf in ("pooled", "ci_lo", "ci_hi"):
        matrix[f"{_name}__{_suf}"] = np.nan
_hdr_cache = {}
for _i, _r in matrix.iterrows():
    if int(_r["folds_used"]) == 0:
        continue
    _key = (_r["architecture"], _r["feature_set"])
    if _key not in _hdr_cache:
        _run, _folds = _headline_run(*_key)
        _hdr_cache[_key] = (_run, _folds, _inner_from_trials(_run, _folds) if _run else {})
    _run, _folds, _inner = _hdr_cache[_key]
    if _run is None:
        continue
    _sk = _inner_skill(
        _fold_frames(_r["feature_set"], _r["architecture"], int(_r["buffer_km"]), _run, _folds),
        _inner,
    )
    for _name in EXTRA_METRICS:
        _pt, _lo, _hi = _sk[_name]
        matrix.loc[_i, f"{_name}__pooled"] = _pt
        matrix.loc[_i, f"{_name}__ci_lo"] = _lo
        matrix.loc[_i, f"{_name}__ci_hi"] = _hi

# Coordinate-only control (a spatial null model), loaded dynamically; skipped if the run is absent.
_coord = sorted((paths.results / "spatial_baseline").glob("*__xy_coords"))
if _coord:
    _cr = _coord[-1]
    _cfolds = [f for f in range(1, 7) if (_cr / f"parcel_predictions_fold{f}.parquet").is_file()]
    _cframes = {f: pd.read_parquet(_cr / f"parcel_predictions_fold{f}.parquet") for f in _cfolds}
    _cpf = pd.read_csv(_cr / "per_fold_metrics.csv")
    _cpf = _cpf[_cpf["level"] == "parcel"]
    _cinner = {int(x["outer_fold"]): float(x["inner_threshold"]) for _, x in _cpf.iterrows()}
    _crow = metric_matrix_row(_cframes, _cinner, _ba, reps=_reps, seed=_seed, n_jobs=8)
    _crow.update({"feature_set": "coordinate_only", "architecture": "xgboost", "buffer_km": 0})
    for _name, (_pt, _lo, _hi) in _inner_skill(_cframes, _cinner).items():
        _crow[f"{_name}__pooled"], _crow[f"{_name}__ci_lo"], _crow[f"{_name}__ci_hi"] = (
            _pt,
            _lo,
            _hi,
        )
    matrix = pd.concat([matrix, pd.DataFrame([_crow])], ignore_index=True)
    print("[augment] coordinate-only control added; accuracy/TSS/MCC columns computed.")

_prev = float(_lab["ogf"].mean())
print(
    f"[note] accuracy/TSS/MCC are at the nested-CV threshold; accuracy is prevalence-sensitive "
    f"(no-skill = {1 - _prev:.2f}), so prefer TSS/MCC and PR-AUC when comparing configurations."
)

## Point 1 - the EO signal over the baseline is real, large and survives the buffer

In [ ]:
def metric_table(frame, label_cols, metrics=METRICS):
    """Compact 'pooled [ci]' display table over the eight metric families."""
    out = frame[label_cols].copy()
    out["folds"] = frame["folds_used"].astype(int).astype(str) + "/6"
    for metric in metrics:
        out[metric] = [
            fmt(p, lo, hi)
            for p, lo, hi in zip(
                frame[f"{metric}__pooled"],
                frame[f"{metric}__ci_lo"],
                frame[f"{metric}__ci_hi"],
                strict=True,
            )
        ]
    return out


print("[Table 1] Performance matrix - pooled PR-AUC etc. with 95% interval (parcel level)")
display1 = metric_table(
    matrix.sort_values(["feature_set", "architecture", "buffer_km"]),
    ["feature_set", "architecture", "buffer_km"],
    metrics=MATRIX_METRICS,
)
print(display1.to_string(index=False))

FOLD_COLS = [f"fold{i}" for i in range(1, 7)]


def per_fold_table(frame, label_cols, metrics=None):
    """Per-fold metric values (no CI): one row per (config, metric); folds 1-6 + SD/range/mean.

    ``metrics`` defaults to METRICS; pass MATRIX_METRICS for the extended families."""
    rows = []
    for _, r in frame.iterrows():
        for metric in metrics or METRICS:
            folds = np.array([r[f"{metric}__fold{i}"] for i in range(1, 7)], dtype=float)
            row = {c: r[c] for c in label_cols}
            row["metric"] = metric
            for i in range(1, 7):
                row[f"fold{i}"] = folds[i - 1]
            row["sd"] = r[f"{metric}__fold_sd"]
            row["range"] = (
                np.nan if np.all(np.isnan(folds)) else np.nanmax(folds) - np.nanmin(folds)
            )
            row["mean"] = r[f"{metric}__fold_mean"]
            rows.append(row)
    out = pd.DataFrame(rows)
    for col in [*FOLD_COLS, "sd", "range", "mean"]:
        out[col] = [fmt(v) for v in out[col]]
    return out


print(
    "\n[Table 1b] Per-fold performance matrix (parcel level) - no CI; "
    "folds 1-6 with SD, range and mean"
)
display1b = per_fold_table(
    matrix.sort_values(["feature_set", "architecture", "buffer_km"]),
    ["feature_set", "architecture", "buffer_km"],
)
print(display1b.to_string(index=False))

print("\n[Table 2] Feature set minus baseline (XGBoost) - difference [95% interval]")
fs_show = metric_table(fs_contrasts, ["contrast", "buffer_km"])
fs_show["pr_auc_p"] = fs_contrasts["pr_auc__p"].round(4)
fs_show["reject"] = fs_contrasts["pr_auc__p"] < 0.05
print(
    fs_show[["contrast", "buffer_km", "folds", "pr_auc", "pr_auc_p", "reject"]].to_string(
        index=False
    )
)

In [ ]:
_OFFSET = 0.18


def _label_point(ax, value, y, *, above, label_pad=(2, -4)):
    """Print the value to 2 dp just outside the marker.

    ``label_pad`` is the (above, below) offset in points; tighten where rows sit close."""
    ax.annotate(
        f"{value:+.2f}" if value < 0 else f"{value:.2f}",
        (value, y),
        textcoords="offset points",
        xytext=(0, label_pad[0] if above else label_pad[1]),
        ha="center",
        va="bottom" if above else "top",
        fontsize=6.5,
        color="0.25",
    )


def points_panel(
    ax,
    frame,
    groups,
    group_labels,
    *,
    metric="pr_auc",
    title="",
    xlabel="",
    buffer_km=10,
    arms=None,
    offset=_OFFSET,
    label_pad=(2, -4),
):
    """Pooled point + 95% interval per group; within-AOI above, spatially-gapped below.

    ``buffer_km`` selects the spatially-gapped arm (10 or 20 km). ``arms`` selects which buffers
    are *shown* (default both: 0 and ``buffer_km``). An omitted arm is still drawn fully
    transparent so the axes autoscale identically - the within-AOI-only figure lands in the exact
    same position as the full one, for a slide-by-slide reveal of the gapped points.
    """
    arms = (0, buffer_km) if arms is None else arms
    y = np.arange(len(groups))
    for sign, arm_km in ((1, 0), (-1, buffer_km)):
        visible = arm_km in arms
        colour = ARM_STYLE[arm_km][0]
        sub = frame[frame["buffer_km"] == arm_km].set_index("_group")
        for yi, group in zip(y, groups, strict=True):
            if group not in sub.index or np.isnan(sub.loc[group, f"{metric}__pooled"]):
                continue
            value = sub.loc[group, f"{metric}__pooled"]
            lo, hi = sub.loc[group, f"{metric}__ci_lo"], sub.loc[group, f"{metric}__ci_hi"]
            yp = yi + sign * offset
            ax.errorbar(
                [value],
                [yp],
                xerr=[[value - lo], [hi - value]],
                fmt=ARM_MARKER[arm_km],
                color=colour,
                ecolor=colour,
                ms=3.3,
                capsize=3,
                lw=1.3,
                alpha=1.0 if visible else 0.0,
            )
            if visible:
                _label_point(ax, value, yp, above=sign > 0, label_pad=label_pad)
    ax.set_yticks(y)
    ax.set_yticklabels(group_labels)
    ax.set_ylim(-0.6, len(groups) - 0.4)
    ax.set_xlabel(xlabel)
    ax.set_title(title, loc="left")
    ax.spines[["top", "right"]].set_visible(False)


def forest_panel(
    ax,
    contrasts,
    labels,
    *,
    title="",
    xlabel="",
    label_fn=contrast_label,
    buffer_km=10,
    arms=None,
    offset=_OFFSET,
    zero_line=True,
    show_significance=True,
    star_style=None,
    star_offset=0.02,
):
    """Paired within-AOI / spatially-gapped PR-AUC contrast differences as a forest plot.

    ``label_fn`` maps a contrast key to its y-axis label (default: the short two-line form).
    ``buffer_km`` selects the spatially-gapped arm (10 or 20 km). ``arms`` selects which buffers
    are *shown* (default both: 0 and ``buffer_km``); an omitted arm is drawn fully transparent so
    the axes autoscale identically (see ``points_panel``). ``show_significance`` draws a ``*``
    beside contrasts whose two-sided bootstrap p is below 0.05; set it False where the interval-vs-zero reading is
    enough and the markers would only add clutter. ``star_style`` overrides the star's text
    properties (Fig. 4 uses a small, unbolded, grey star) and ``star_offset`` its gap from the
    upper interval bound in data units.
    """
    star_kw = {"fontsize": 10, "fontweight": "bold", **(star_style or {})}
    arms = (0, buffer_km) if arms is None else arms
    y = np.arange(len(labels))
    for sign, arm_km in ((1, 0), (-1, buffer_km)):
        visible = arm_km in arms
        colour = ARM_STYLE[arm_km][0]
        sub = contrasts[contrasts["buffer_km"] == arm_km].set_index("contrast")
        for yi, label in zip(y, labels, strict=True):
            if label not in sub.index or np.isnan(sub.loc[label, "pr_auc__pooled"]):
                continue
            value = sub.loc[label, "pr_auc__pooled"]
            lo, hi = sub.loc[label, "pr_auc__ci_lo"], sub.loc[label, "pr_auc__ci_hi"]
            yp = yi + sign * offset
            ax.errorbar(
                [value],
                [yp],
                xerr=[[value - lo], [hi - value]],
                fmt=ARM_MARKER[arm_km],
                color=colour,
                ecolor=colour,
                ms=3.3,
                capsize=3,
                lw=1.3,
                alpha=1.0 if visible else 0.0,
            )
            if visible:
                _label_point(ax, value, yp, above=sign > 0)
                if show_significance and bool(sub.loc[label].get("pr_auc__p", 1.0) < 0.05):
                    ax.text(hi + star_offset, yp, "*", va="center", **star_kw)
    if zero_line:
        ax.axvline(0.0, color="0.6", ls="-", lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels([label_fn(label) for label in labels])
    ax.set_ylim(-0.6, len(labels) - 0.4)
    ax.set_xlabel(xlabel)
    ax.set_title(title, loc="left")
    ax.spines[["top", "right"]].set_visible(False)


def point1_figure(arms=None, buffer_km=10):
    """Point-1 figure at ``buffer_km``: per-feature-set PR-AUC + feature-set contrasts."""
    fig, (ax_l, ax_r) = plt.subplots(
        1,
        2,
        figsize=get_figure_size("double", aspect=0.4),
        constrained_layout=True,
        gridspec_kw={
            "width_ratios": [1.0, 1.25]
        },  # wider right panel for the full bracketed labels
    )
    mx = matrix[matrix["architecture"] == "xgboost"].assign(_group=lambda d: d["feature_set"])
    points_panel(
        ax_l,
        mx,
        FS_ORDER,
        [per_config_label(fs) for fs in FS_ORDER],
        title="a) Per-feature-set PR-AUC (XGBoost)",
        xlabel="Pooled PR-AUC",
        buffer_km=buffer_km,
        arms=arms,
    )
    forest_panel(
        ax_r,
        fs_contrasts,
        [f"{fs} - baseline" for fs in FS_ORDER if fs != "baseline"],
        title="b) Feature set minus baseline",
        xlabel="PR-AUC difference (A minus B)",
        label_fn=contrast_label_full,
        buffer_km=buffer_km,
        arms=arms,
    )
    arm_legend(fig, buffer_km=buffer_km)
    return fig

## Point 2 - the EO representations are indistinguishable once spatial dependence is controlled

In [ ]:
emb_show = metric_table(emb_contrasts, ["contrast", "buffer_km"])
emb_show["pr_auc_p"] = emb_contrasts["pr_auc__p"].round(4)
print("[Point 2] EO-representation pairwise contrasts (XGBoost), PR-AUC difference [interval]")
print(emb_show[["contrast", "buffer_km", "folds", "pr_auc", "pr_auc_p"]].to_string(index=False))

emb_labels = list(dict.fromkeys(emb_contrasts["contrast"]))


def point2_figure(arms=None, buffer_km=10):
    """Point-2 figure at ``buffer_km``: per-feature-set PR-AUC + EO pairwise contrasts."""
    mx = matrix[matrix["architecture"] == "xgboost"].assign(_group=lambda d: d["feature_set"])
    fig, (ax_l, ax_r) = plt.subplots(
        1,
        2,
        figsize=get_figure_size("double", aspect=0.4),
        constrained_layout=True,
        gridspec_kw={
            "width_ratios": [1.0, 1.25]
        },  # wider right panel for the full bracketed labels
    )
    points_panel(
        ax_l,
        mx,
        FS_ORDER,
        [per_config_label(fs) for fs in FS_ORDER],
        xlabel="Pooled PR-AUC",
        buffer_km=buffer_km,
        arms=arms,
    )
    forest_panel(
        ax_r,
        emb_contrasts,
        emb_labels,
        xlabel="PR-AUC difference (A minus B)",
        label_fn=contrast_label_full,
        buffer_km=buffer_km,
        arms=arms,
    )
    arm_legend(fig, buffer_km=buffer_km)
    return fig

In [ ]:
# Shared by the combined points-1/2 figure and Fig. 4 below: the PR-AUC no-skill floor, the
# coordinate-only control as matrix rows and as a paired contrast against Baseline, and the
# row-label / panel-title helpers.
# PR-AUC no-skill baseline: the parcel prevalence of the pooled out-of-fold evaluation set.
# Verified identical (0.2091 on the same 4,825 parcels) for every architecture, every feature
# set and both arms -- the spatial buffer changes the training set, not the test folds -- so a
# single constant serves all four panels.
_prev_run = sorted((paths.results / "main_nested_cv").glob("*__xgboost__baseline_tessera"))[-1]
PARCEL_PREVALENCE = float(
    pd.concat(
        [pd.read_parquet(p) for p in sorted(_prev_run.glob("parcel_predictions_fold*.parquet"))]
    )["y_true"].mean()
)

# Feature sets top-to-bottom in the panels; reversed for plotting (y=0 is the bottom row).
FS_ROWS = ["baseline", "baseline_conventional_eo", "baseline_alphaearth", "baseline_tessera"]
MATRIX_OFFSET = 0.12  # arm separation: close together but not overlapping


def _stack_label(fs):
    """Two-line stack name: 'Baseline\n(topography + access)', 'Baseline\n+ TESSERA', etc."""
    if fs == "baseline":
        return "Baseline\n(topography + access)"
    if fs == "xy_coords":
        return "Coordinate-only\ncontrol (x, y)"
    return f"Baseline\n+ {SHORT_FS[fs]}"


def _center_yticklabels(ax):
    """Centre every tick label on its tick.

    Single-line labels previously used "center_baseline", which centres the text baseline
    rather than the glyphs and so rode high next to the two-line labels beside them."""
    for tick in ax.get_yticklabels():
        tick.set_va("center")


def _panel_title(ax, main, subtitle=None):
    """Left-aligned panel title; an optional italic second line sits just above the axes."""
    ax.annotate(
        main,
        xy=(0, 1),
        xycoords="axes fraction",
        xytext=(0, 13 if subtitle else 2),
        textcoords="offset points",
        ha="left",
        va="bottom",
        fontsize=8.5,
    )
    if subtitle:
        ax.annotate(
            subtitle,
            xy=(0, 1),
            xycoords="axes fraction",
            xytext=(0, 2),
            textcoords="offset points",
            ha="left",
            va="bottom",
            fontsize=8.5,
            style="italic",
        )


def _coord_matrix_rows():
    """Coordinate-only control as matrix rows (buffer 0 = within AOI, 10 = spatially-gapped).

    XGBoost comes from the xy_coords gap sweep. Each CNN has its own measured pair, written by
    scripts/run_coordinate_cnn_buffer.py; an architecture without one falls back to the XGBoost
    value, which is the theoretical expectation (a convolution over coordinate channels is
    fixed by the centre pixel) but is flagged in the raw CSV via `measured`."""
    coord = gap_detail_by_fs.get("xy_coords")
    if coord is None:
        return None
    xgb = {}
    for buffer_km in (0, 10):
        stat = coord[
            (coord["arm"] == "buffered")
            & (coord["metric"] == "pr_auc")
            & (coord["gap_km"] == buffer_km)
        ].set_index("stat")["value"]
        if "pooled" not in stat.index or pd.isna(stat["pooled"]):
            return None
        xgb[buffer_km] = {
            "pr_auc__pooled": stat["pooled"],
            "pr_auc__ci_lo": stat.get("ci_lo", float("nan")),
            "pr_auc__ci_hi": stat.get("ci_hi", float("nan")),
        }

    measured = {}
    buffer_dir = paths.results / "coordinate_cnn_buffer"
    if buffer_dir.is_dir():
        for path in sorted(buffer_dir.glob("cnn_*.csv")):
            table = pd.read_csv(path)
            for _, r in table.iterrows():
                measured[(str(r["architecture"]), int(r["buffer_km"]))] = {
                    "pr_auc__pooled": float(r["pr_auc__pooled"]),
                    "pr_auc__ci_lo": float(r["pr_auc__ci_lo"]),
                    "pr_auc__ci_hi": float(r["pr_auc__ci_hi"]),
                }

    out = []
    for buffer_km in (0, 10):
        for arch in ARCH_ORDER:
            values = measured.get((arch, buffer_km))
            out.append(
                {
                    "feature_set": "xy_coords",
                    "architecture": arch,
                    "buffer_km": buffer_km,
                    "measured": values is not None or arch == "xgboost",
                    **(values if values is not None else xgb[buffer_km]),
                }
            )
    return pd.DataFrame(out)


coord_matrix_rows = _coord_matrix_rows()


def _coord_baseline_contrast():
    """Paired (coordinate-only minus Baseline) contrast at the 0 and 10 km buffered arms.

    Built with utils.reporting.paired_row (the helper behind feature_set_contrasts), so the
    row is computed exactly like the existing baseline contrasts: the parcel predictions at
    each gap come from the fold checkpoints of the two buffered gap sweeps, the inner
    thresholds from each feature set's headline run (as run_gap_analysis scores them), and
    the paired block bootstrap uses the shared block set. None until both sweeps exist."""
    import json

    from utils.bootstrap import load_block_assignment
    from utils.reporting import inner_thresholds_from_per_fold, paired_row
    from utils.terminology import BOOTSTRAP_REPS_CROSS_CONFIG, FOLD_IDS, SEED

    def checkpoint(run, fold):
        return run / "fold_checkpoints" / f"fold{fold}_parcel.parquet"

    def buffered_run(fs):
        """Newest complete sweep of ``fs`` whose checkpoints carry the buffered arm."""
        for run in reversed(gap_runs_by_fs.get(fs, [])):
            path = checkpoint(run, FOLD_IDS[0])
            if (
                path.is_file()
                and (pd.read_parquet(path, columns=["arm"])["arm"] == "buffered").any()
            ):
                return run
        return None

    def frames(run, gap_km):
        out = {}
        for fold in FOLD_IDS:
            path = checkpoint(run, fold)
            if not path.is_file():
                continue
            table = pd.read_parquet(path)
            arm = table[(table["arm"] == "buffered") & (table["gap_km"] == gap_km)]
            if len(arm):
                out[fold] = arm.drop(columns=["arm", "gap_km", "n_train"]).reset_index(drop=True)
        return out

    def inner_thresholds(run):
        source = paths.repo_root / json.loads((run / "run_metadata.json").read_text())["source_run"]
        return inner_thresholds_from_per_fold(pd.read_csv(source / "per_fold_metrics.csv"))

    xy_run, base_run = buffered_run("xy_coords"), buffered_run("baseline")
    if xy_run is None or base_run is None:
        return None
    block = load_block_assignment(paths.labels / "ogf_reference_labels_partitioned.gpkg")
    inner_xy, inner_base = inner_thresholds(xy_run), inner_thresholds(base_run)
    rows = []
    for gap_km in (0, 10):
        stats = paired_row(
            frames(xy_run, gap_km),
            frames(base_run, gap_km),
            inner_xy,
            inner_base,
            block,
            reps=BOOTSTRAP_REPS_CROSS_CONFIG,
            seed=SEED,
            n_jobs=8,
        )
        rows.append(
            {
                "contrast": "xy_coords - baseline",
                "set_a": "xy_coords",
                "set_b": "baseline",
                "buffer_km": gap_km,
                "folds_used": int(stats.pop("folds_used")),
                "pr_auc__p_holm": np.nan,
                "pr_auc__reject_holm": False,
                **stats,
            }
        )
    return pd.DataFrame(rows)


coord_baseline_contrast = _coord_baseline_contrast()

### Points 1 and 2 in one figure

In [ ]:
emb_labels = list(dict.fromkeys(emb_contrasts["contrast"]))
p12_emb_labels = list(reversed(emb_labels))  # Fig. 4 order (AlphaEarth - conventional EO on top)
# Panel rows in the order of Fig. 4 (bottom-to-top: TESSERA ... Baseline, coordinate-only control).
# The control's 0/10 km PR-AUC comes from the xy_coords gap sweep (coord_matrix_rows) and its
# paired contrast against Baseline from coord_baseline_contrast, both computed above for Fig. 4.
P12_ROWS = [*reversed(FS_ROWS), "xy_coords"]


def _p12_contrast_label(key):
    """Second-panel label: the feature set (or the control) contrasted against Baseline."""
    set_a, _set_b = key.split(" - ")
    return "Coordinate-only\ncontrol (x, y)" if set_a == "xy_coords" else _short(set_a)


def points1and2_figure(arms=None, buffer_km=10):
    """Combined points 1 & 2 figure (three panels) at ``buffer_km``.

    ``arms`` selects the buffers shown (default both). An omitted arm is drawn fully transparent
    (see ``points_panel``), so the 0 km-only version keeps every point exactly where the two-arm
    figure has it. Titles, axis labels and row labels follow Fig. 4.
    """
    mx = matrix[(matrix["architecture"] == "xgboost") & matrix["feature_set"].isin(FS_ORDER)]
    rows = list(reversed(FS_ROWS))
    if coord_matrix_rows is not None:
        mx = pd.concat(
            [mx, coord_matrix_rows[coord_matrix_rows["architecture"] == "xgboost"]],
            ignore_index=True,
        )
        rows = P12_ROWS
    mx = mx.assign(_group=lambda d: d["feature_set"])
    contrast_rows = [f"{fs} - baseline" for fs in reversed(FS_ROWS) if fs != "baseline"]
    fs_used = fs_contrasts
    if coord_baseline_contrast is not None:
        contrast_rows = [*contrast_rows, "xy_coords - baseline"]
        fs_used = pd.concat([fs_contrasts, coord_baseline_contrast], ignore_index=True)
    fig, (ax1, ax2, ax3) = plt.subplots(
        1, 3, figsize=get_figure_size("double", aspect=0.378), constrained_layout=True
    )
    points_panel(
        ax1,
        mx,
        rows,
        [_stack_label(fs) for fs in rows],
        xlabel="Pooled PR-AUC",
        offset=0.11,
        buffer_km=buffer_km,
        arms=arms,
    )
    forest_panel(
        ax2,
        fs_used,
        contrast_rows,
        xlabel="ΔPR-AUC",
        label_fn=_p12_contrast_label,
        zero_line=False,
        offset=0.11,
        buffer_km=buffer_km,
        arms=arms,
    )
    forest_panel(
        ax3,
        emb_contrasts,
        p12_emb_labels,
        xlabel="ΔPR-AUC",
        label_fn=contrast_label_full,
        zero_line=False,
        offset=0.11,
        buffer_km=buffer_km,
        arms=arms,
    )
    for ax, title, subtitle in (
        (ax1, "(a) XGBoost", " "),  # blank subtitle keeps the three titles on one line
        (ax2, "(b) Baseline contrasts", "(XGBoost)"),
        (ax3, "(c) EO contrasts", "(XGBoost)"),
    ):
        _panel_title(ax, title, subtitle)
        ax.grid(axis="x", color="0.85", lw=0.5)  # vertical reference grid behind the points
        ax.set_axisbelow(True)
        ax.tick_params(length=2.0, pad=2.0)  # smaller ticks, labels closer to them
        _center_yticklabels(ax)
    # Reference rules as in Fig. 4: the no-skill floor in (a), a black zero line in (b) and (c).
    ax1.axvline(PARCEL_PREVALENCE, color="black", ls=(0, (5, 3)), lw=0.9, zorder=1)
    ax1.set_xlim(left=min(ax1.get_xlim()[0], PARCEL_PREVALENCE - 0.04))
    for ax in (ax2, ax3):
        ax.axvline(0.0, color="black", ls="-", lw=0.9, zorder=1)
    ax2.xaxis.set_major_locator(MultipleLocator(0.2))  # the control row widens the range
    # One legend row with fixed slots (no buffer | buffered | no skill), as in Fig. 4. In the
    # 0 km-only version the buffered entry is hidden rather than dropped, so the other two
    # entries keep their positions and the two figures overlay exactly.
    shown = (0, buffer_km) if arms is None else tuple(arms)
    handles = [
        Line2D(
            [0], [0], marker=ARM_MARKER[km], color=ARM_STYLE[km][0], lw=0, label=ARM_STYLE[km][1]
        )
        for km in (0, buffer_km)
    ] + [
        Line2D(
            [0],
            [0],
            color="black",
            ls=(0, (5, 3)),
            lw=0.9,
            label=f"No skill (OGF prevalence = {PARCEL_PREVALENCE:.2f})",
        )
    ]
    legend = fig.legend(handles=handles, loc="outside lower center", ncol=3, frameon=False)
    legend_handles = getattr(legend, "legend_handles", None) or legend.legendHandles
    for km, handle, text in zip((0, buffer_km), legend_handles, legend.get_texts(), strict=False):
        if km not in shown:
            handle.set_visible(False)
            text.set_alpha(0.0)
    return fig


save_if_enabled(
    points1and2_figure(buffer_km=10),
    f"{NOTEBOOK}/eo_signal_and_representation_contrasts",
    data=emb_contrasts,
)
# The same figure with the 10 km arm hidden (drawn transparent, so nothing moves).
save_if_enabled(
    points1and2_figure(arms=(0,), buffer_km=10),
    f"{NOTEBOOK}/eo_signal_and_representation_contrasts_0km",
    data=emb_contrasts,
)

## Point 3 - spatial context gives no parcel-level gain over the pixel model

In [ ]:
arch_show = metric_table(arch_contrasts, ["contrast", "buffer_km"])
arch_show["pr_auc_p"] = arch_contrasts["pr_auc__p"].round(4)
print("[Table 3] CNN minus XGBoost (per feature set), PR-AUC difference [interval]")
print(arch_show[["contrast", "buffer_km", "folds", "pr_auc", "pr_auc_p"]].to_string(index=False))


def point3_figure(fs, arms=None, buffer_km=10):
    """Point-3 figure at ``buffer_km`` for one feature set (arch PR-AUC + contrasts)."""
    fig, (ax_l, ax_r) = plt.subplots(
        1, 2, figsize=get_figure_size("double", aspect=0.4), constrained_layout=True
    )
    sub = matrix[matrix["feature_set"] == fs].assign(_group=lambda d: d["architecture"])
    points_panel(
        ax_l,
        sub,
        ARCH_ORDER,
        [SHORT_ARCH[a] for a in ARCH_ORDER],
        xlabel="Pooled PR-AUC",
        buffer_km=buffer_km,
        arms=arms,
    )
    forest_panel(
        ax_r,
        arch_contrasts,
        [f"{a}__{fs} - xgboost__{fs}" for a in ARCH_ORDER if a != "xgboost"],
        xlabel="PR-AUC difference (A minus B)",
        buffer_km=buffer_km,
        arms=arms,
    )
    arm_legend(fig, buffer_km=buffer_km)
    return fig


def point3_prauc_absolute(fs, arms=None, buffer_km=10):
    """Single-column figure: pooled parcel PR-AUC (absolute, no contrast) per architecture."""
    fig, ax = plt.subplots(figsize=get_figure_size("single", aspect=0.85), constrained_layout=True)
    sub = matrix[matrix["feature_set"] == fs].assign(_group=lambda d: d["architecture"])
    arch_rows = list(reversed(ARCH_ORDER))  # XGBoost at top, then CNN 3x3, 5x5, 7x7 going down
    points_panel(
        ax,
        sub,
        arch_rows,
        [SHORT_ARCH[a] for a in arch_rows],
        title=f"Architecture: {FEATURE_SETS[fs]}",
        xlabel="PR-AUC",
        offset=0.11,
        buffer_km=buffer_km,
        arms=arms,
    )
    ax.grid(axis="x", color="0.85", lw=0.5)
    ax.set_axisbelow(True)
    ax.tick_params(length=2.0, pad=2.0)  # smaller ticks, labels closer to them
    arm_legend(
        fig, buffer_km=buffer_km, ncol=2, columnspacing=0.6, handlelength=1.3, handletextpad=0.4
    )
    return fig


save_if_enabled(
    point3_prauc_absolute("baseline_tessera"),
    f"{NOTEBOOK}/architecture_comparison_tessera",
    data=matrix[matrix["feature_set"] == "baseline_tessera"],
)

## Comprehensive PR-AUC matrix

In [ ]:
def prauc_matrix_figure(coord_as_row=False):
    """Row 1: PR-AUC per architecture (both arms). Row 2: the three contrast families.

    Laid out by hand (in inches) rather than by constrained_layout so every panel is exactly
    the same square size, row 2 aligns to row 1's outer edges, and its middle panel is centred.
    """
    show_coord_rows = coord_as_row and coord_matrix_rows is not None
    rows = list(reversed(FS_ROWS)) + (["xy_coords"] if show_coord_rows else [])

    # Coordinate-only control as dashed reference rules (pooled PR-AUC of the xy_coords
    # buffered arm at 0 and 10 km) - unless coord_as_row, when it is a data row instead.
    coord_ref = {}
    if not coord_as_row and (coord := gap_detail_by_fs.get("xy_coords")) is not None:
        for _km in (0, 10):
            _sel = coord[
                (coord["arm"] == "buffered")
                & (coord["metric"] == "pr_auc")
                & (coord["stat"] == "pooled")
                & (coord["gap_km"] == _km)
            ]
            if len(_sel):
                coord_ref[_km] = float(_sel["value"].iloc[0])

    fig_w = get_figure_size("double")[0]
    # The panels sit flush with the right edge (right margin only clears the last x tick
    # label), and the reclaimed width goes into the left margin so row 2's first panel can
    # show its full contrast labels on two lines. panel_scale is set to hold the panel size
    # constant while left grows.
    left, right = 1.28, 0.02
    # Panels stay square, so once the paddings are tight the only remaining way to lose
    # height is to shrink them; the column gap absorbs whatever width they give up.
    panel_scale = 0.9612
    pw = panel_scale * (fig_w - left - right - 3 * 0.135) / 4
    gap = (fig_w - left - right - 4 * pw) / 3
    row1_xaxis, row2_xaxis = 0.36, 0.36
    # Overall height is held constant across both versions; the dashed-line layout with its
    # two-row legend sets the reference height.
    fig_h = 0.17 + pw + row1_xaxis + 0.02 + 0.31 + pw + row2_xaxis + 0.46
    if show_coord_rows:
        # Row 1 carries five stacks x two arms, so it takes every inch the layout can spare:
        # the legend sits in a thin band flush with the bottom edge and row 2 keeps its
        # height, leaving row 1 the residual so its rows are spaced further apart.
        # Fixed overall height, so the foot band and the two row heights trade against each
        # other. legend_h sets how far the (bottom-flush) legend sits clear of the x-axis
        # labels; the rest is split so neither row's value labels collide.
        top_pad, row_gap, bot_pad, legend_h = 0.15, 0.01, 0.0, 0.16
        row2_title, ph2 = 0.31, 1.1 * pw
        ph = fig_h - (
            top_pad + row1_xaxis + row_gap + row2_title + ph2 + row2_xaxis + legend_h + bot_pad
        )
    else:
        ph = ph2 = pw
        top_pad, row_gap, bot_pad = 0.17, 0.02, 0.0
        row2_title, legend_h = 0.31, (0.46 if coord_ref else 0.24)
        fig_h = (
            top_pad + ph + row1_xaxis + row_gap + row2_title + ph2 + row2_xaxis + legend_h + bot_pad
        )
    row2_bottom = bot_pad + legend_h + row2_xaxis
    row1_bottom = row2_bottom + ph2 + row2_title + row_gap + row1_xaxis
    row1_right = left + 4 * pw + 3 * gap  # right edge of the fourth row-1 panel

    # The coord-row variant needs the no-skill floor to read against five data rows, so it is
    # a black dashed rule there; the dashed-line variant keeps the subtle solid grey.
    no_skill_style = (
        {"color": "black", "ls": (0, (5, 3)), "lw": 0.9}
        if show_coord_rows
        else {"color": "0.5", "ls": "-", "lw": 0.8}
    )

    fig = plt.figure(figsize=(fig_w, fig_h))

    def _axes(x_in, y_in, height=None):
        h = ph if height is None else height
        return fig.add_axes([x_in / fig_w, y_in / fig_h, pw / fig_w, h / fig_h])

    # Row 1 - one panel per architecture, both arms together, y labels only on the first.
    axes_top = []
    for i, arch in enumerate(ARCH_ORDER):
        ax = _axes(left + i * (pw + gap), row1_bottom)
        mx = matrix[matrix["architecture"] == arch]
        if show_coord_rows:
            mx = pd.concat(
                [mx, coord_matrix_rows[coord_matrix_rows["architecture"] == arch]],
                ignore_index=True,
            )
        mx = mx.assign(_group=lambda d: d["feature_set"])
        points_panel(
            ax,
            mx,
            rows,
            [_stack_label(fs) for fs in rows],
            xlabel="Pooled PR-AUC",
            offset=MATRIX_OFFSET,
            buffer_km=10,
            label_pad=(2, -3),
        )
        # zorder 1 keeps these reference rules above the grid but below the data.
        ax.axvline(PARCEL_PREVALENCE, zorder=1, **no_skill_style)
        for _km in (0, 10):
            if _km in coord_ref:
                ax.axvline(
                    coord_ref[_km], color=ARM_STYLE[_km][0], ls=(0, (6, 3)), lw=1.0, zorder=0
                )
        _panel_title(ax, f"({chr(97 + i)}) {TITLE_ARCH[arch]}")
        if i:
            ax.tick_params(labelleft=False)
        ax.xaxis.labelpad = 2.5
        axes_top.append(ax)

    # Row 2 - same square panels: first flush left with row 1, third flush right, second centred.
    # Panel 3's labels are short ('minus [XGBoost]'), so it needs only a narrow margin; the slack
    # goes to panel 2, whose labels are the widest in the figure. Panel 1 keeps the figure's
    # left margin and wraps its long names instead (see wrap_at below).
    row2_label_gap = 0.62
    row2_lefts = [
        left,
        row1_right - 2 * pw - row2_label_gap,
        row1_right - pw,
    ]
    # Panel (e) gains the coordinate-only contrast in the coord-row variant: it is negative
    # (the control is well below Baseline) with an interval clear of zero, so the panel shows
    # Baseline significantly above the control and significantly below the EO stacks at once.
    show_coord_contrast = show_coord_rows and coord_baseline_contrast is not None
    fs_contrasts_used = (
        pd.concat([fs_contrasts, coord_baseline_contrast], ignore_index=True)
        if show_coord_contrast
        else fs_contrasts
    )
    contrast_specs = [
        (
            fs_contrasts_used,
            [f"{fs} - baseline" for fs in reversed(FS_ROWS) if fs != "baseline"]
            + (["xy_coords - baseline"] if show_coord_contrast else []),
            contrast_label_full,
            "Baseline contrasts",
            "(XGBoost)",
        ),
        (
            emb_contrasts,
            list(reversed(list(dict.fromkeys(emb_contrasts["contrast"])))),
            contrast_label_full,
            "EO contrasts",
            "(XGBoost)",
        ),
        (
            arch_contrasts,
            [
                f"{a}__baseline_tessera - xgboost__baseline_tessera"
                for a in reversed(ARCH_ORDER)
                if a != "xgboost"
            ],
            contrast_label_full,
            "Architecture contrasts",
            "(baseline + TESSERA)",
        ),
    ]
    axes_bot = []
    for panel_i, (x_in, (frame, labels, label_fn, title, subtitle)) in enumerate(
        zip(row2_lefts, contrast_specs, strict=True)
    ):
        ax = _axes(x_in, row2_bottom, height=ph2)
        # The baseline contrasts are wholly positive, so the y axis sits exactly on zero and is
        # the zero reference itself; the other two families straddle zero and carry a solid
        # zero rule.
        on_zero = panel_i == 0 and not show_coord_contrast
        forest_panel(
            ax,
            frame,
            labels,
            xlabel="ΔPR-AUC",
            label_fn=label_fn,
            offset=MATRIX_OFFSET,
            buffer_km=10,
            zero_line=False,
            # subtle star beside contrasts whose two-sided bootstrap p is below 0.05
            star_style={"fontsize": 6.5, "fontweight": "normal", "color": "0.3"},
            star_offset=0.008,
        )
        if show_coord_rows:
            ax.axvline(0.0, color="black", ls="-", lw=0.9, zorder=1)
        ax.tick_params(axis="y", labelsize=6.5)
        if on_zero:
            ax.set_xlim(left=0.0)
        _panel_title(ax, f"({chr(97 + len(ARCH_ORDER) + panel_i)}) {title}", subtitle)
        ax.xaxis.labelpad = 1.5
        axes_bot.append(ax)

    for ax in [*axes_top, *axes_bot]:
        if show_coord_rows:
            ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        ax.grid(axis="x", color="0.85", lw=0.5)
        ax.set_axisbelow(True)
        ax.tick_params(length=2.0, pad=2.0)
        _center_yticklabels(ax)
    # One shared PR-AUC scale across the architecture row: the left edge keeps the no-skill
    # line in view, the right edge stops 0.01 above the largest interval bound so no width is
    # wasted beyond the widest whisker.
    lo = min(ax.get_xlim()[0] for ax in axes_top)
    hi = max(ax.get_xlim()[1] for ax in axes_top)
    pad = 0.03 * (hi - lo)
    top_rows = matrix[
        matrix["architecture"].isin(ARCH_ORDER)
        & matrix["feature_set"].isin(FS_ROWS)
        & matrix["buffer_km"].isin([0, 10])
    ]
    right = float(top_rows["pr_auc__ci_hi"].max()) + 0.01
    for ax in axes_top:
        ax.set_xlim(lo - pad, right)

    # The EO and architecture contrasts each scale to their own data, but symmetrically about
    # zero: both families straddle zero, so a centred axis lets the eye read the sign and the
    # magnitude of an effect fairly. The baseline contrasts keep their own asymmetric range,
    # which is dominated by the coordinate-only row reaching past -0.24.
    for ax in axes_bot[1:]:
        limit = max(abs(bound) for bound in ax.get_xlim())
        ax.set_xlim(-limit, limit)

    # Legend (column-major, ncol=3): within-AOI, spatially-gapped and the no-skill floor; the
    # dashed-line version also carries the two coordinate-only reference entries (second row).
    handles = [
        Line2D([0], [0], marker=ARM_MARKER[0], color=ARM_STYLE[0][0], lw=0, label=ARM_STYLE[0][1]),
        Line2D(
            [0], [0], marker=ARM_MARKER[10], color=ARM_STYLE[10][0], lw=0, label=ARM_STYLE[10][1]
        ),
    ]
    for _km in (0, 10):
        if _km in coord_ref:
            handles.append(
                Line2D(
                    [0],
                    [0],
                    color=ARM_STYLE[_km][0],
                    ls=(0, (6, 3)),
                    lw=1.0,
                    label=f"Coordinate-only control ({_km} km buffer)",
                )
            )
    handles.append(
        Line2D(
            [0],
            [0],
            **no_skill_style,
            label=f"No skill (OGF prevalence = {PARCEL_PREVALENCE:.2f})",
        )
    )
    fig.legend(
        handles=handles,
        loc="lower center",
        # bot_pad is the layout margin; the extra shift closes the few points of white
        # that the legend's own text metrics leave under its ink.
        bbox_to_anchor=(0.5, (bot_pad - 0.04) / fig_h),
        ncol=3,
        frameon=False,
        borderpad=0.0,  # drop the legend's internal padding so it sits flush with the foot
    )
    return fig


# The coordinate-only control is drawn as a data row (not a dashed reference rule), so this
# is the only version of the figure.
_coord_row_data = matrix[matrix["buffer_km"].isin([0, 10])]
if coord_matrix_rows is not None:
    _coord_row_data = pd.concat(
        [_coord_row_data, coord_matrix_rows[coord_matrix_rows["buffer_km"].isin([0, 10])]],
        ignore_index=True,
    )
save_if_enabled(
    prauc_matrix_figure(coord_as_row=True),
    f"{NOTEBOOK}/fig_4_prauc_by_feature_set_and_architecture",
    data=_coord_row_data,
)

## Why 10 km - the spatial-buffer justification

In [ ]:
def starve_from(counts):
    """Gap (km) at which the worst fold first keeps <20% of its gap-0 pool, and that fold."""
    buffered = counts[counts["arm"] == "buffered"].assign(
        rf=lambda d: d["n_retained"] / d["n_train_parcels"]
    )
    retention = buffered.pivot_table(index="outer_fold", columns="gap_km", values="rf")
    gaps = sorted(retention.columns)
    starved = [g for g in gaps if retention[g].min() < STARVE_THRESHOLD]
    collapse_fold = int(retention[max(gaps)].idxmin())
    return (starved[0] if starved else max(gaps)), collapse_fold


if gap_detail is None:
    print("gap_detail.csv not in this spatial_buffer run; re-run scripts/run_gap_analysis.py.")
    GAPS = STARVED_FROM = COLLAPSE_FOLD = None
else:
    GAPS = sorted(gap_detail["gap_km"].unique())
    STARVED_FROM, COLLAPSE_FOLD = starve_from(gap_counts)
    buffered = gap_detail[gap_detail["arm"] == "buffered"]
    table4 = buffered.pivot_table(index=["metric", "stat"], columns="gap_km", values="value")
    table4 = table4.reindex(
        [(m, s) for m in METRICS for s in ("pooled", "ci_lo", "ci_hi", "fold_mean")]
    )
    buf_counts = gap_counts[gap_counts["arm"] == "buffered"]
    ret = buf_counts.assign(rf=lambda d: 100 * d["n_retained"] / d["n_train_parcels"])
    print(
        f"[Table 4] XGBoost + TESSERA gap sweep (buffered arm), gaps {GAPS} km; "
        f"folds collapse from {STARVED_FROM} km"
    )
    print(table4.round(3).to_string())
    print("\nTraining retained (% of gap-0 pool), fold mean:")
    print(ret.groupby("gap_km")["rf"].mean().round(1).to_string())
    print("Training prevalence (% old-growth), fold mean:")
    print(
        (buf_counts.groupby("gap_km")["n_train_parcels_prevalence"].mean() * 100)
        .round(1)
        .to_string()
    )

In [ ]:
def gap_series(detail, arm, metric, stat):
    sel = detail[(detail["arm"] == arm) & (detail["metric"] == metric) & (detail["stat"] == stat)]
    return sel.set_index("gap_km")["value"].sort_index()


def starved_span(ax, *, annotate=True):
    """Shade the gap range over which folds have begun to collapse, labelled at its top."""
    if STARVED_FROM is not None and STARVED_FROM < max(GAPS):
        ax.axvspan(STARVED_FROM, max(GAPS), color="0.5", alpha=0.12, lw=0)
        if annotate:
            ax.annotate(
                "Training-data collapse",
                xy=((STARVED_FROM + max(GAPS)) / 2, 1.0),
                xycoords=("data", "axes fraction"),
                ha="center",
                va="top",
                fontsize=6.5,
                color="0.35",
            )


# Feature sets drawn on the performance-change axis (headline TESSERA first). Each feature
# set's buffered arm is compared against its OWN matched-random control, so every line is 0
# at gap 0 and isolates the spatial-buffer effect; a line appears once both its arms exist.
PERF_CHANGE_FS = (
    "baseline",
    "baseline_conventional_eo",
    "baseline_alphaearth",
    "baseline_tessera",
    "xy_coords",
)


def gap_performance_change(fs):
    """Feature set `fs`'s buffered arm minus its OWN matched-random control (pooled parcel
    PR-AUC by gap). Empty until that feature set's control arm has been computed."""
    detail = gap_detail_by_fs[fs]
    buffered = gap_series(detail, "buffered", "pr_auc", "pooled")
    control = gap_series(detail, "control", "pr_auc", "pooled")
    if buffered.empty or control.empty:
        return buffered.iloc[:0]
    return buffered - control.reindex(buffered.index)


def plot_gap_performance_change(
    ax, *, legend=True, shade=True, xmax=None, annotate=True, label_map=None
):
    """Draw buffered-minus-control PR-AUC versus gap for every available feature set.

    With ``xmax`` set, the data and the x-axis both stop at that gap (km); endpoint markers
    are drawn unclipped so they stay whole at the axis edge. ``label_map`` overrides the
    per-feature-set legend labels; ``annotate`` labels the collapse band at its top."""
    labels = label_map if label_map is not None else GAP_FS_LABEL
    if shade:
        starved_span(ax, annotate=annotate)
    ax.axhline(0, color="0.4", lw=0.6)
    ax.axvline(10, color="0.4", ls=":", lw=1)
    for fs in PERF_CHANGE_FS:
        if fs not in gap_detail_by_fs:
            continue
        change = gap_performance_change(fs)
        if change.empty:  # buffered present but its own control not computed yet
            continue
        if xmax is not None:
            change = change[change.index <= xmax]
        ax.plot(
            change.index,
            change.to_numpy(),
            ls="-",
            marker=GAP_FS_MARKER[fs],
            color=GAP_FS_COLOUR[fs],
            lw=1.1,
            ms=2.6,
            label=labels[fs],
            clip_on=xmax is None,
            # the coordinate-only control is context, so it sits behind the stacks
            zorder=1.5 if fs == "xy_coords" else 2.0,
        )
    ax.set_xlabel("Spatial buffer gap (km)")
    ax.set_ylabel(" Difference in PR-AUC\n(buffered \u2212 control)")
    ax.spines[["top", "right"]].set_visible(False)
    if xmax is not None:
        ax.set_xlim(0, xmax)
    if legend:
        ax.legend(frameon=False, fontsize=7)


def gap_performance_change_table():
    """Long-form buffered / own-control / difference PR-AUC per (feature set, gap) behind the
    axis; only feature sets with both arms are included."""
    cols = ["gap_km", "feature_set", "buffered_pr_auc", "control_pr_auc", "performance_change"]
    frames = []
    for fs in PERF_CHANGE_FS:
        if fs not in gap_detail_by_fs:
            continue
        detail = gap_detail_by_fs[fs]
        buffered = gap_series(detail, "buffered", "pr_auc", "pooled")
        control = gap_series(detail, "control", "pr_auc", "pooled")
        if buffered.empty or control.empty:
            continue
        ctrl = control.reindex(buffered.index)
        frames.append(
            pd.DataFrame(
                {
                    "gap_km": buffered.index,
                    "feature_set": fs,
                    "buffered_pr_auc": buffered.to_numpy(),
                    "control_pr_auc": ctrl.to_numpy(),
                    "performance_change": (buffered - ctrl).to_numpy(),
                }
            )
        )
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=cols)


def _split_legend(fig, handles, labels, *, ncol=2, fontsize=8):
    """Legend as a tight ncol grid filled row by row, with the coordinate-only entry last.

    An "outside" legend cannot be filled row-major, so the constrained-layout rect reserves a
    band at the foot of the figure and the legend is centred inside it by hand. Reading
    left-to-right, top-to-bottom, the coordinate-only control lands in the bottom-left cell,
    flush with the stacks' left column."""
    pairs = list(zip(handles, labels, strict=True))
    solo = [p for p in pairs if p[1].startswith("Coordinate-only")]
    grid = [p for p in pairs if not p[1].startswith("Coordinate-only")] + solo
    nrows = -(-len(grid) // ncol)
    band = 0.075 * nrows  # figure fraction per legend row
    fig.get_layout_engine().set(rect=(0, band, 1, 1 - band))
    style = {
        "frameon": False,
        "fontsize": fontsize,
        "columnspacing": 0.8,
        "handletextpad": 0.4,
        "handlelength": 1.2,
        "borderaxespad": 0.0,
    }
    hs = _legend_rowmajor([p[0] for p in grid], ncol)
    ls = _legend_rowmajor([p[1] for p in grid], ncol)
    fig.legend(hs, ls, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=ncol, **style)


def _legend_rowmajor(seq, ncol):
    """Reorder legend entries so matplotlib's column-major fill reads left-to-right,
    top-to-bottom (last entry lands at the bottom, not mid-column)."""
    n = len(seq)
    nrows = -(-n // ncol)
    out = [None] * n
    read = 0
    for row in range(nrows):
        for col in range(ncol):
            idx = col * nrows + row
            if idx < n:
                out[idx] = seq[read]
                read += 1
    return out


if gap_detail is not None:
    # Full sweep (0-30 km) with the training-data-collapse shading; the collapse region is
    # labelled on the grey band.
    fig, ax = plt.subplots(figsize=get_figure_size("single", aspect=0.85), constrained_layout=True)
    # This figure spells out what the baseline stack contains; the shared GAP_FS_LABEL
    # (also used by prauc_vs_buffer) keeps the plain name.
    plot_gap_performance_change(
        ax,
        legend=False,
        label_map={**GAP_FS_LABEL, "baseline": "Baseline (topography + access)"},
    )
    handles, labels = ax.get_legend_handles_labels()
    _split_legend(fig, handles, labels)
    save_if_enabled(
        fig, f"{NOTEBOOK}/fig_5_performance_change_vs_buffer", data=gap_performance_change_table()
    )

In [ ]:
# Absolute pooled parcel PR-AUC versus spatial buffer gap: each feature set's buffered arm
# (solid, markers) and its own matched-random control (dashed), in the feature set's colour.
# Companion to performance_change_vs_buffer, which plots buffered minus control. A feature set
# appears once it has both arms. The grey band marks training-data collapse.
if gap_detail is not None:
    fig, ax = plt.subplots(
        figsize=(
            get_figure_size("double", aspect=1.0)[0],
            get_figure_size("single", aspect=1.0)[1],
        ),
        constrained_layout=True,
    )
    starved_span(ax)
    ax.axhline(PARCEL_PREVALENCE, color="0.5", ls="-", lw=0.8, zorder=0)
    ax.axvline(10, color="0.4", ls=":", lw=1)
    fs_handles = []
    for fs in PERF_CHANGE_FS:
        detail = gap_detail_by_fs.get(fs)
        if detail is None:
            continue
        buffered = gap_series(detail, "buffered", "pr_auc", "pooled")
        control = gap_series(detail, "control", "pr_auc", "pooled")
        if buffered.empty or control.empty:
            continue
        colour = GAP_FS_COLOUR[fs]
        ax.plot(
            buffered.index,
            buffered.to_numpy(),
            ls="-",
            marker=GAP_FS_MARKER[fs],
            color=colour,
            lw=1.1,
            ms=2.6,
        )
        ax.plot(control.index, control.to_numpy(), "--", color=colour, lw=1.1)
        fs_handles.append(
            Line2D(
                [],
                [],
                color=colour,
                lw=1.1,
                marker=GAP_FS_MARKER[fs],
                ms=2.6,
                label=GAP_FS_LABEL[fs],
            )
        )
    ax.set_xlabel("Spatial buffer gap (km)")
    ax.set_ylabel("Parcel PR-AUC (pooled OOF)")
    ax.spines[["top", "right"]].set_visible(False)
    style_handles = [
        Line2D([], [], color="0.3", lw=1.1, label="Buffered"),
        Line2D([], [], color="0.3", lw=1.1, ls="--", label="Control"),
        Line2D([], [], color="0.5", lw=0.8, label=f"No skill ({PARCEL_PREVALENCE:.2f})"),
    ]
    legend_handles = fs_handles + style_handles
    legend_ncol = min(len(legend_handles), 4)
    fig.legend(
        # row-major, so the coordinate-only control sits bottom-left under the stacks
        handles=_legend_rowmajor(legend_handles, legend_ncol),
        loc="outside lower center",
        ncol=legend_ncol,
        frameon=False,
        fontsize=8,
        columnspacing=1.0,
        handletextpad=0.4,
        handlelength=1.4,
    )
    save_if_enabled(
        fig,
        f"{NOTEBOOK}/fig_s7_prauc_vs_buffer",
        data=gap_performance_change_table(),
        bbox_inches="tight",
        pad_inches=0.02,
    )

In [ ]:
# Table: how far the buffered arm falls below its matched control at the 10 km headline buffer
# and the 20 km robustness buffer, per feature set, against how much training data the buffer
# removes. "delta PR-AUC" is buffered - control (pooled parcel PR-AUC; negative = below control);
# "perf drop %" is 100 * (control - buffered) / control; "train lost %" is the fold-mean share of
# the gap-0 training parcels the buffer excludes (buffer geometry, so identical across feature
# sets). The "10-20 km" group is the min/max delta PR-AUC over every buffer in that inclusive
# range (more negative = larger drop). A feature-set row appears once it has both arms.
PERF_DROP_KM = (10, 20)
PERF_DROP_BAND = (10, 20)  # inclusive km range for the min/max drop across all buffers
if gap_detail is not None:
    buf_counts = gap_counts[gap_counts["arm"] == "buffered"]
    train_lost = (
        buf_counts.assign(lost=100 * buf_counts["n_excluded"] / buf_counts["n_train_parcels"])
        .groupby("gap_km")["lost"]
        .mean()
    )
    lo, hi = PERF_DROP_BAND
    drop_rows = {}
    for fs in PERF_CHANGE_FS:
        detail = gap_detail_by_fs.get(fs)
        if detail is None:
            continue
        buffered = gap_series(detail, "buffered", "pr_auc", "pooled")
        control = gap_series(detail, "control", "pr_auc", "pooled")
        if buffered.empty or control.empty:
            continue
        rec = {}
        for km in PERF_DROP_KM:
            if km not in buffered.index or km not in control.index:
                continue
            b = float(buffered.loc[km])
            c = float(control.loc[km])
            rec[(f"{km} km", "delta PR-AUC")] = b - c
            rec[(f"{km} km", "perf drop %")] = 100 * (c - b) / c
            rec[(f"{km} km", "train lost %")] = float(train_lost.get(km, float("nan")))
        change = buffered - control.reindex(buffered.index)
        band = change[(change.index >= lo) & (change.index <= hi)].dropna()
        if not band.empty:
            rec[(f"{lo}-{hi} km", "min delta PR-AUC")] = float(band.min())
            rec[(f"{lo}-{hi} km", "mean delta PR-AUC")] = float(band.mean())
            rec[(f"{lo}-{hi} km", "max delta PR-AUC")] = float(band.max())
        drop_rows[SHORT_FS[fs]] = rec
    perf_drop_table = pd.DataFrame(drop_rows).T
    perf_drop_table.index.name = "feature set"
    print("[Table 5] Buffered performance drop vs training data removed, at 10 km and 20 km:")
    print(perf_drop_table.round(3).to_string())

In [ ]:
def prevalence_band(counts, arm):
    """(mean, min, max) over folds of the training prevalence, gap-indexed."""
    piv = counts[counts["arm"] == arm].pivot_table(
        index="outer_fold", columns="gap_km", values="n_train_parcels_prevalence"
    )
    return piv.mean(), piv.min(), piv.max()


if gap_detail is not None:
    fig, (ax_a, ax_b, ax_c) = plt.subplots(
        1, 3, figsize=get_figure_size("double", aspect=0.3), constrained_layout=True
    )

    # (a) performance change from removing nearby training data: each feature set's buffered
    #     arm minus the reused control (pooled parcel PR-AUC). Negative = below the control.
    plot_gap_performance_change(
        ax_a,
        legend=False,
        annotate=False,
        # the same legend labels as Fig. 5; too wide for the panel, so the legend goes below
        label_map={**GAP_FS_LABEL, "baseline": "Baseline (topography + access)"},
    )
    ax_a.set_title("a) Performance change", loc="left", fontsize=9)
    fig.legend(
        *ax_a.get_legend_handles_labels(),
        loc="outside lower center",
        ncol=3,
        frameon=False,
        fontsize=7,
        columnspacing=1.2,
        handletextpad=0.5,
    )

    # (b) training data surviving the buffer: fold-range band and the fold mean.
    starved_span(ax_b, annotate=False)
    buf_counts = gap_counts[gap_counts["arm"] == "buffered"].assign(
        rf=lambda d: 100 * d["n_retained"] / d["n_train_parcels"]
    )
    retention = buf_counts.pivot_table(index="outer_fold", columns="gap_km", values="rf")
    ax_b.fill_between(
        GAPS,
        retention.min().reindex(GAPS).to_numpy(),
        retention.max().reindex(GAPS).to_numpy(),
        color=GAP_BUFFERED,
        alpha=0.15,
        lw=0,
    )
    ax_b.plot(
        GAPS,
        retention.mean().reindex(GAPS).to_numpy(),
        "-o",
        color=GAP_BUFFERED,
        lw=1.1,
        ms=2.5,
        label="Fold mean",
    )
    # Trace the collapsing fold's own trajectory (it drives the band's lower edge to zero).
    ax_b.plot(
        GAPS,
        retention.loc[COLLAPSE_FOLD].reindex(GAPS).to_numpy(),
        "--",
        color="#c0392b",
        lw=1.1,
        label=f"Fold {COLLAPSE_FOLD} (collapses)",
    )
    ax_b.axvline(10, color="0.4", ls=":", lw=1)
    ax_b.set_xlabel("Spatial buffer gap (km)")
    ax_b.set_ylabel("Training retained\n(% of gap-0 pool)")
    ax_b.set_title("b) Surviving training data", loc="left", fontsize=9)
    ax_b.set_ylim(0, 100)
    handles, labels = ax_b.get_legend_handles_labels()
    ax_b.legend(
        [*handles, Patch(facecolor="0.5", alpha=0.2)],
        [*labels, "Training-data collapse"],
        frameon=False,
        fontsize=7,
        loc="upper right",
    )
    ax_b.spines[["top", "right"]].set_visible(False)

    # (c) training prevalence by arm, with fold-range bands and the constant overall prevalence.
    starved_span(ax_c, annotate=False)
    overall = gap_counts[gap_counts["gap_km"] == 0]["n_train_parcels_prevalence"].mean()
    for arm, colour in (("buffered", GAP_BUFFERED), ("control", GAP_CONTROL)):
        mean, lo, hi = prevalence_band(gap_counts, arm)
        ax_c.fill_between(mean.index, lo.to_numpy(), hi.to_numpy(), color=colour, alpha=0.15, lw=0)
        ax_c.plot(
            mean.index, mean.to_numpy(), "-o", color=colour, lw=1.1, ms=2.5, label=arm.capitalize()
        )
    ax_c.axhline(overall, color="0.4", ls=":", lw=1.2, label=f"Overall ({overall:.2f})")
    ax_c.axvline(10, color="0.4", ls=":", lw=1)
    ax_c.set_xlabel("Spatial buffer gap (km)")
    ax_c.set_ylabel("Training prevalence\n(OGF fraction)")
    ax_c.set_title("c) Training prevalence by arm", loc="left", fontsize=9)
    ax_c.set_ylim(0, 0.5)
    ax_c.legend(frameon=False, loc="upper left", fontsize=7)
    ax_c.spines[["top", "right"]].set_visible(False)

    save_if_enabled(
        fig,
        f"{NOTEBOOK}/fig_s6_spatial_buffer_justification",
        data={
            "counts": gap_counts,
            "performance_change": gap_performance_change_table(),
        },
    )

## XGBoost feature importance by feature set

In [ ]:
EMBED = ("TESSERA", "AlphaEarth")
GROUP_COLOUR = {
    "Terrain": PALETTE_CATEGORICAL["orange"],
    "Access": PALETTE_CATEGORICAL["yellow"],
    "Sentinel-2 optical": "#1B7A3D",  # dark green, paired with the lighter vegetation indices
    "Sentinel-1 SAR": "#8c564b",
    "Vegetation indices": PALETTE_CATEGORICAL["light_green"],  # NDVI + phenology (VPP)
    "TESSERA": PALETTE_CATEGORICAL["blue"],
    "AlphaEarth": PALETTE_CATEGORICAL["magenta"],
}
READABLE = {
    "elevation_m": "Elevation",
    "slope_deg": "Slope",
    "heat_load_index": "Heat load index",
    "dist_paved_road_m": "Dist. to paved road",
    "dist_unpaved_road_m": "Dist. to unpaved road",
    "dist_footpath_m": "Dist. to footpath",
    "s2_blue": "Sentinel-2 blue",
    "s2_green": "Sentinel-2 green",
    "s2_red": "Sentinel-2 red",
    "s2_nir": "Sentinel-2 NIR",
    "swir_b11": "SWIR band 11",
    "swir_b12": "SWIR band 12",
    "ndvi_p90": "NDVI p90",
    "ndvi_p50": "NDVI p50",
    "ndvi_p10": "NDVI p10",
    "s1_vv": "Sentinel-1 VV",
    "s1_vh": "Sentinel-1 VH",
    "s1_vh_vv_ratio": "Sentinel-1 VH/VV ratio",
    "vpp_ampl": "VPP amplitude",
    "vpp_eosd": "VPP EOS date",
    "vpp_eosv": "VPP EOS value",
    "vpp_lslope": "VPP left slope",
    "vpp_maxv": "VPP max value",
    "vpp_minv": "VPP min value",
    "vpp_rslope": "VPP right slope",
    "vpp_sosd": "VPP SOS date",
    "vpp_sosv": "VPP SOS value",
    "vpp_sprod": "VPP productivity",
}


def feat_group(band):
    if band in ("elevation_m", "slope_deg", "heat_load_index"):
        return "Terrain"
    if band.startswith("dist_"):
        return "Access"
    if band.startswith("vpp_") or band.startswith("ndvi_"):
        return "Vegetation indices"
    if band.startswith("tessera_"):
        return "TESSERA"
    if band.startswith("alphaearth_"):
        return "AlphaEarth"
    if band.startswith("s1_"):
        return "Sentinel-1 SAR"
    return "Sentinel-2 optical"


def mean_importance(run_dir):
    folds = [
        pd.read_csv(f).set_index("feature")["importance"]
        for f in sorted(run_dir.glob("importance_fold*.csv"))
    ]
    return pd.concat(folds, axis=1).mean(axis=1)


FS_ORDER = ["baseline", "baseline_conventional_eo", "baseline_alphaearth", "baseline_tessera"]
FS_TITLE = {
    "baseline": "Baseline",
    "baseline_conventional_eo": "Baseline + conventional EO",
    "baseline_alphaearth": "Baseline + AlphaEarth",
    "baseline_tessera": "Baseline + TESSERA",
}
runs = {}
for fs in FS_ORDER:
    cands = [
        c
        for c in sorted((paths.results / "main_nested_cv").glob(f"*__xgboost__{fs}"))
        if list(c.glob("importance_fold*.csv"))
    ]
    if cands:
        runs[fs] = cands[-1]

if not runs:
    print("No feature-importance CSVs found; run scripts/run_nested_cv.py.")
else:
    per_fs = {}
    embed_dims = {}  # grp -> number of embedding dimensions (for the legend)
    raw_importance = []  # long-form per-feature gain behind every bar (for the raw CSV)
    for fs, run_dir in runs.items():
        imp = mean_importance(run_dir)
        raw_importance.extend(
            {
                "feature_set": fs,
                "feature": b,
                "group": feat_group(b),
                "label": READABLE.get(b, b.replace("_", " ").capitalize()),
                "importance": float(imp[b]),
            }
            for b in imp.index
        )
        bars = []  # (label, total, group, dims-or-None)
        for grp in EMBED:
            members = [b for b in imp.index if feat_group(b) == grp]
            if members:
                embed_dims[grp] = len(members)
                dims = imp[members].sort_values(ascending=False).to_numpy()
                bars.append((grp, float(imp[members].sum()), grp, dims))
        for b in imp.index:
            if feat_group(b) not in EMBED:
                bars.append(
                    (
                        READABLE.get(b, b.replace("_", " ").capitalize()),
                        float(imp[b]),
                        feat_group(b),
                        None,
                    )
                )
        per_fs[fs] = sorted(bars, key=lambda x: x[1])  # ascending -> largest at the top of barh

    # Two columns at the same overall width: the three compact feature sets stacked on the
    # left, the long conventional-EO list on its own on the right. Narrower sub-plots than the
    # single-column stack, and a shorter figure driven by the taller of the two columns.
    COLUMNS = [
        ["baseline", "baseline_alphaearth", "baseline_tessera"],
        ["baseline_conventional_eo"],
    ]
    columns = [[fs for fs in col if fs in runs] for col in COLUMNS]
    columns = [col for col in columns if col]
    col_counts = [sum(len(per_fs[fs]) for fs in col) for col in columns]

    width = get_figure_size("double", aspect=1.0)[0]
    fig = plt.figure(
        figsize=(width, 0.675 * (0.19 * max(col_counts) + 1.6)), constrained_layout=True
    )
    fig.set_constrained_layout_pads(w_pad=0.0, h_pad=0.04, wspace=0.01, hspace=0.02)
    outer = fig.add_gridspec(1, len(columns))
    axes_by_fs = {}
    for c, col in enumerate(columns):
        inner = outer[0, c].subgridspec(len(col), 1, height_ratios=[len(per_fs[fs]) for fs in col])
        for r, fs in enumerate(col):
            axes_by_fs[fs] = fig.add_subplot(inner[r, 0])
    bottom_of_column = {col[-1] for col in columns}  # last feature set in each column

    for fs, ax in axes_by_fs.items():
        bars = per_fs[fs]
        for y, (_label, total, grp, dims) in enumerate(bars):
            if dims is None:
                ax.barh(
                    y, total, height=0.72, color=GROUP_COLOUR[grp], edgecolor="white", linewidth=0.3
                )
            else:  # aggregated embedding: a stack of its dims with alternating shading
                left = 0.0
                for i, imp_i in enumerate(dims):
                    ax.barh(
                        y,
                        imp_i,
                        left=left,
                        height=0.72,
                        color=GROUP_COLOUR[grp],
                        alpha=0.9 if i % 2 else 0.5,
                        edgecolor="none",
                    )
                    left += imp_i
        ax.set_yticks(range(len(bars)))
        ax.set_yticklabels([b[0] for b in bars], fontsize=6.5)
        ax.set_ylim(-0.6, len(bars) - 0.4)
        ax.set_title(f"({chr(97 + FS_ORDER.index(fs))}) {FS_TITLE[fs]}", loc="left", fontsize=9)
        ax.set_xlim(left=0)
        ax.grid(axis="x", color="0.85", lw=0.5)
        ax.set_axisbelow(True)
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        if fs in bottom_of_column:
            ax.set_xlabel("Gain importance (mean over folds)", fontsize=8)
    # Fixed legend order; each GFM embedding shows its alternating dark/light
    # shading as two equal-width swatches.
    legend_order = [
        "Terrain",
        "Access",
        "Sentinel-1 SAR",
        "Sentinel-2 optical",
        "Vegetation indices",
        "AlphaEarth",
        "TESSERA",
    ]
    handles, labels = [], []
    for grp in legend_order:
        colour = GROUP_COLOUR[grp]
        if grp in embed_dims:
            handles.append((Patch(facecolor=colour, alpha=0.9), Patch(facecolor=colour, alpha=0.5)))
            labels.append(f"{grp} ({embed_dims[grp]} embeddings)")
        else:
            handles.append(Patch(facecolor=colour))
            labels.append(grp)
    fig.legend(
        handles=handles,
        labels=labels,
        loc="outside lower center",
        ncol=len(handles),
        frameon=False,
        fontsize=7,
        columnspacing=1.0,
        handlelength=1.1,
        handletextpad=0.4,
        handler_map={tuple: HandlerTuple(ndivide=2, pad=0.0)},
    )
    save_if_enabled(
        fig,
        f"{NOTEBOOK}/fig_6_feature_importance_by_predictor_stack",
        data=pd.DataFrame(raw_importance),
    )

In [ ]:
# Hyperparameter-tuning trial spread on the tuning objective (mean parcel PR-AUC over the
# inner folds, the `pr_auc` column of hp_trials_fold*.csv). Per config x outer fold we take
# the worst, median and selected (best) trial and report three gaps:
#   median - worst   : how far a typical draw sits above the worst
#   selected - worst : full search spread (what selecting the best buys over the worst)
#   selected - median: gain of the selected config over a typical draw
# Each is summarised as median [Q1, Q3] across config x fold combinations, for XGBoost and
# the CNNs (3x3/5x5/7x7 pooled), over all feature groups and split by feature group. Latest
# run per (architecture, feature set), COMPLETE trials only.
def _family(arch):
    """Group the three CNN receptive fields into a single 'CNN' family."""
    return "XGBoost" if arch == "xgboost" else "CNN"


def hp_trial_spread_records():
    """worst/median/selected trial gaps per (family, feature set, outer fold)."""
    rows = []
    for arch in ARCH_ORDER:
        for fs in FS_ORDER:
            runs_fs = [
                c
                for c in sorted((paths.results / "main_nested_cv").glob(f"*__{arch}__{fs}"))
                if list(c.glob("hp_trials_fold*.csv"))
            ]
            if not runs_fs:
                continue
            for trials_path in sorted(runs_fs[-1].glob("hp_trials_fold*.csv")):
                pr_auc = pd.read_csv(trials_path).query("state == 'COMPLETE'")["pr_auc"].to_numpy()
                if pr_auc.size == 0:
                    continue
                worst, med, selected = pr_auc.min(), np.median(pr_auc), pr_auc.max()
                rows.append(
                    {
                        "family": _family(arch),
                        "feature_set": fs,
                        "median - worst": float(med - worst),
                        "selected - worst": float(selected - worst),
                        "selected - median": float(selected - med),
                    }
                )
    return pd.DataFrame(rows)


GAP_COLUMNS = ["median - worst", "selected - worst", "selected - median"]


def _median_iqr(values):
    """'median [Q1, Q3]' of a numeric Series, to 3 dp."""
    return f"{values.median():.3f} [{values.quantile(0.25):.3f}, {values.quantile(0.75):.3f}]"


hp_spread = hp_trial_spread_records()
if hp_spread.empty:
    print("No hp_trials_fold*.csv under results/main_nested_cv; run scripts/run_nested_cv.py.")
else:
    summary_rows = []
    for family in ("XGBoost", "CNN"):
        fam = hp_spread[hp_spread["family"] == family]
        groups = [("all feature groups", fam)] + [
            (SHORT_FS[fs], fam[fam["feature_set"] == fs]) for fs in FS_ORDER
        ]
        for label, sub in groups:
            if sub.empty:
                continue
            summary_rows.append(
                {"family": family, "feature group": label, "n": len(sub)}
                | {gap: _median_iqr(sub[gap]) for gap in GAP_COLUMNS}
            )
    print(
        "[HP tuning] inner-fold parcel PR-AUC trial gaps per config x fold,\n"
        "            median [Q1, Q3] across config x fold combinations\n"
    )
    print(pd.DataFrame(summary_rows).to_string(index=False))

In [ ]:
# Save every printed table in this notebook as a CSV under
# figures/009_performance_matrix_and_buffers/tables/.
_table_dir = paths.figures / NOTEBOOK / "tables"
_table_dir.mkdir(parents=True, exist_ok=True)
_TABLES = {
    "table1_performance_matrix": "display1",
    "table1b_per_fold": "display1b",
    "table2_feature_set_vs_baseline": "fs_show",
    "eo_representation_contrasts": "emb_show",
    "table3_cnn_vs_xgboost": "arch_show",
    "table4_tessera_gap_sweep": "table4",
    "table5_buffered_performance_drop": "perf_drop_table",
}
_saved = []
for _name, _var in _TABLES.items():
    _df = globals().get(_var)
    if isinstance(_df, pd.DataFrame):
        _keep = _df.index.name is not None or isinstance(_df.index, pd.MultiIndex)
        _df.to_csv(_table_dir / f"{_name}.csv", index=_keep)
        _saved.append(_name)
if isinstance(globals().get("summary_rows"), list):
    pd.DataFrame(summary_rows).to_csv(_table_dir / "buffer_change_summary.csv", index=False)
    _saved.append("buffer_change_summary")
print(f"[tables] saved {len(_saved)} table CSVs to {_table_dir}: {', '.join(_saved)}")